# Predicción de Abandono de Clientes en Telecomunicaciones

## Introducción

El presente proyecto tiene como objetivo analizar información histórica de clientes de una empresa de telecomunicaciones para identificar patrones relacionados con el abandono de clientes, conocido como Customer Churn.

El conjunto de datos contiene información sobre los servicios contratados por los clientes, características demográficas, información de sus cuentas, antigüedad, tipo de contrato, método de pago y cargos asociados.

La variable principal del análisis es `Churn`, la cual indica si un cliente abandonó el servicio durante el último mes.

A través de las etapas de comprensión del negocio, comprensión de los datos, preparación, análisis exploratorio y evaluación de aspectos éticos, se busca generar una base de datos adecuada para el desarrollo de un modelo de Machine Learning capaz de identificar clientes con mayor riesgo de abandono.

## 1. Importación de librerías

In [8]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

#si da error
#!pip install plotly

## 2. Carga del conjunto de datos

En esta sección se carga el conjunto de datos original y se realiza una primera revisión de su contenido.

In [9]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Comprensión inicial de los datos

In [10]:
print("Número de filas:", df.shape[0])
print("Número de columnas:", df.shape[1])
df.info()

Número de filas: 7043
Número de columnas: 21
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 1

## 4. Evaluación de calidad de los datos

En esta sección se evalúa la calidad del conjunto de datos mediante la identificación de valores faltantes, registros duplicados, inconsistencias en los tipos de datos, valores únicos y posibles valores inválidos.

Esta etapa permite justificar las decisiones de limpieza y preparación realizadas antes del análisis exploratorio y del futuro proceso de modelamiento.

##4.1 Revisar valores faltantes

In [11]:
missing_values = df.isnull().sum()

print(missing_values)

missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    "Valores faltantes": missing_values,
    "Porcentaje (%)": missing_percentage
})

missing_summary

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


,Valores faltantes,Porcentaje (%)
customerID,0,0.0
gender,0,0.0
SeniorCitizen,0,0.0
Partner,0,0.0
Dependents,0,0.0
tenure,0,0.0
PhoneService,0,0.0
MultipleLines,0,0.0
InternetService,0,0.0
OnlineSecurity,0,0.0


##4.2 Buscar valores vacíos o espacios

In [12]:
empty_spaces = {}

for column in df.select_dtypes(include="object").columns:
    empty_spaces[column] = (
        df[column].astype(str).str.strip().eq("")
    ).sum()
empty_spaces




{'customerID': np.int64(0),
 'gender': np.int64(0),
 'Partner': np.int64(0),
 'Dependents': np.int64(0),
 'PhoneService': np.int64(0),
 'MultipleLines': np.int64(0),
 'InternetService': np.int64(0),
 'OnlineSecurity': np.int64(0),
 'OnlineBackup': np.int64(0),
 'DeviceProtection': np.int64(0),
 'TechSupport': np.int64(0),
 'StreamingTV': np.int64(0),
 'StreamingMovies': np.int64(0),
 'Contract': np.int64(0),
 'PaperlessBilling': np.int64(0),
 'PaymentMethod': np.int64(0),
 'TotalCharges': np.int64(11),
 'Churn': np.int64(0)}

In [13]:
empty_spaces_df = pd.DataFrame(
    list(empty_spaces.items()),
    columns=["Columna", "Valores vacíos o espacios"]
)

empty_spaces_df

,Columna,Valores vacíos o espacios
0,customerID,0
1,gender,0
2,Partner,0
3,Dependents,0
4,PhoneService,0
5,MultipleLines,0
6,InternetService,0
7,OnlineSecurity,0
8,OnlineBackup,0
9,DeviceProtection,0


##4.3 Registros duplicados

In [14]:
duplicates = df.duplicated().sum()

print("Cantidad de registros duplicados:", duplicates)

duplicate_percentage = (duplicates / len(df)) * 100

print(f"Porcentaje de registros duplicados: {duplicate_percentage:.2f}%")

Cantidad de registros duplicados: 0
Porcentaje de registros duplicados: 0.00%


##4.4 Revisar identificadores únicos


In [15]:
print("Total de registros:", len(df))
print("Clientes únicos:", df["customerID"].nunique())
print("customerID duplicados:", df["customerID"].duplicated().sum())

Total de registros: 7043
Clientes únicos: 7043
customerID duplicados: 0


##4.5 Revisar valores únicos y categorías

In [16]:
categorical_columns = df.select_dtypes(include="object").columns

for column in categorical_columns:
    print(f"\n{'='*50}")
    print(f"Variable: {column}")
    print(f"{'='*50}")
    print(df[column].value_counts(dropna=False))


Variable: customerID
customerID
3186-AJIEK    1
7590-VHVEG    1
5575-GNVDE    1
8775-CEBBJ    1
2823-LKABH    1
             ..
6713-OKOMC    1
1452-KIOVK    1
9305-CDSKC    1
9237-HQITU    1
7795-CFOCW    1
Name: count, Length: 7043, dtype: int64

Variable: gender
gender
Male      3555
Female    3488
Name: count, dtype: int64

Variable: Partner
Partner
No     3641
Yes    3402
Name: count, dtype: int64

Variable: Dependents
Dependents
No     4933
Yes    2110
Name: count, dtype: int64

Variable: PhoneService
PhoneService
Yes    6361
No      682
Name: count, dtype: int64

Variable: MultipleLines
MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

Variable: InternetService
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

Variable: OnlineSecurity
OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

Va

##**4.6 TotalCharges**

In [17]:
df_clean = df.copy()

In [18]:
df_clean["TotalCharges"] = (
    df_clean["TotalCharges"]
    .astype(str)
    .str.strip()
)

In [19]:
df_clean["TotalCharges"] = pd.to_numeric(
    df_clean["TotalCharges"],
    errors="coerce"
)

In [20]:
print(
    "Valores nulos después de la conversión:",
    df_clean["TotalCharges"].isnull().sum()
)

Valores nulos después de la conversión: 11


In [21]:
df_clean[
    df_clean["TotalCharges"].isnull()
][
    ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]
]

,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,NaN
753,3115-CZMZD,0,20.25,NaN
936,5709-LVOEQ,0,80.85,NaN
1082,4367-NUYAO,0,25.75,NaN
1340,1371-DWPAZ,0,56.05,NaN
3331,7644-OMVMY,0,19.85,NaN
3826,3213-VVOLG,0,25.35,NaN
4380,2520-SGTTA,0,20.00,NaN
5218,2923-ARZLG,0,19.70,NaN
6670,4075-WKNIU,0,73.35,NaN


### Análisis de los registros afectados

Los 11 registros con valores inválidos en `TotalCharges` presentan una antigüedad (`tenure`) de 0 meses. Esto indica que corresponden a clientes sin antigüedad acumulada en el servicio y que no poseen un valor registrado de cargos totales.

Debido a que estos registros representan una proporción muy pequeña del conjunto de datos y no existe información suficiente para determinar de manera confiable el valor de `TotalCharges`, se optará por excluirlos del conjunto de datos utilizado para el análisis.

In [22]:
invalid_totalcharges = df_clean["TotalCharges"].isnull().sum()

invalid_percentage = (
    invalid_totalcharges / len(df_clean)
) * 100

print("Registros con TotalCharges inválido:", invalid_totalcharges)
print(f"Porcentaje afectado: {invalid_percentage:.2f}%")

Registros con TotalCharges inválido: 11
Porcentaje afectado: 0.16%


### Tratamiento de valores inválidos

Debido a que los registros afectados representan aproximadamente el 0,16% del conjunto de datos, se considera que su eliminación tendrá un impacto mínimo sobre el análisis. Se eliminan estas observaciones para evitar introducir valores artificiales mediante imputación.

In [23]:
rows_before = len(df_clean)

df_clean = df_clean.dropna(
    subset=["TotalCharges"]
).copy()

rows_after = len(df_clean)

print("Registros antes:", rows_before)
print("Registros después:", rows_after)
print("Registros eliminados:", rows_before - rows_after)

Registros antes: 7043
Registros después: 7032
Registros eliminados: 11


## 5. Verificación posterior a la limpieza

Una vez realizado el tratamiento de los valores inválidos, se verifica nuevamente la estructura y calidad del conjunto de datos.

In [24]:
print("===== CALIDAD DESPUÉS DE LA LIMPIEZA =====")

print("\nDimensiones:")
print(df_clean.shape)

print("\nValores nulos totales:")
print(df_clean.isnull().sum().sum())

print("\nRegistros duplicados:")
print(df_clean.duplicated().sum())

print("\nTipo de TotalCharges:")
print(df_clean["TotalCharges"].dtype)

===== CALIDAD DESPUÉS DE LA LIMPIEZA =====

Dimensiones:
(7032, 21)

Valores nulos totales:
0

Registros duplicados:
0

Tipo de TotalCharges:
float64


In [25]:
quality_after_cleaning = pd.DataFrame({
    "Indicador": [
        "Número de registros",
        "Número de variables",
        "Valores faltantes",
        "Registros duplicados"
    ],
    "Resultado": [
        len(df_clean),
        df_clean.shape[1],
        df_clean.isnull().sum().sum(),
        df_clean.duplicated().sum()
    ]
})

quality_after_cleaning

,Indicador,Resultado
0,Número de registros,7032
1,Número de variables,21
2,Valores faltantes,0
3,Registros duplicados,0


In [26]:
print("Filas finales:", df_clean.shape[0])
print("Columnas finales:", df_clean.shape[1])
print("Valores nulos:", df_clean.isnull().sum().sum())
print("Duplicados:", df_clean.duplicated().sum())
print("Tipo TotalCharges:", df_clean["TotalCharges"].dtype)

Filas finales: 7032
Columnas finales: 21
Valores nulos: 0
Duplicados: 0
Tipo TotalCharges: float64


## 6. Análisis Exploratorio de Datos (EDA)

El análisis exploratorio tiene como objetivo identificar patrones, tendencias, distribuciones y relaciones relevantes entre las características de los clientes y la variable objetivo `Churn`.

Se analizarán especialmente los factores que podrían estar asociados al abandono de clientes.

## 6.1 KPI principal: Tasa general de abandono

In [27]:
churn_counts = df_clean["Churn"].value_counts()

churn_rate = (
    df_clean["Churn"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Cantidad de clientes:")
print(churn_counts)

print("\nPorcentaje:")
print(churn_rate)

Cantidad de clientes:
Churn
No     5163
Yes    1869
Name: count, dtype: int64

Porcentaje:
Churn
No     73.42
Yes    26.58
Name: proportion, dtype: float64


In [28]:
churn_rate = (
    df_clean["Churn"]
    .eq("Yes")
    .mean()
) * 100

print(f"Tasa de abandono: {churn_rate:.2f}%")

Tasa de abandono: 26.58%


### Interpretación del KPI

El conjunto de datos contiene 7.032 clientes después del proceso de limpieza. De ellos, 5.163 clientes (73,42%) no abandonaron el servicio, mientras que 1.869 clientes (26,58%) sí lo hicieron.

La tasa general de abandono es de **26,58%**, lo que indica que aproximadamente uno de cada cuatro clientes del conjunto analizado abandonó el servicio.

Este resultado establece una línea base para el proyecto y permite dimensionar el problema de negocio. En las siguientes etapas del análisis se estudiarán las características de los clientes para identificar qué segmentos presentan mayores tasas de abandono y, por lo tanto, podrían requerir estrategias de retención.

Además, se observa una diferencia entre ambas clases de `Churn`, por lo que existe cierto desbalance en la variable objetivo. Esta situación deberá considerarse posteriormente durante el desarrollo y evaluación del modelo de Machine Learning.

In [29]:
fig = px.bar(
    churn_counts.reset_index(),
    x="Churn",
    y="count",
    text="count",
    title="Distribución de clientes según abandono"
)

fig.update_layout(
    xaxis_title="Estado de abandono",
    yaxis_title="Cantidad de clientes"
)

fig.show()

### Interpretación del resultado

La distribución de `Churn` permite dimensionar el problema de abandono de clientes. La tasa de abandono constituye el KPI principal del proyecto, ya que representa la proporción de clientes que dejaron el servicio durante el período analizado.

Este indicador servirá como línea base para comparar posteriormente los distintos segmentos de clientes y determinar qué características presentan una mayor asociación con el abandono.

##6.2 Distribución porcentual del abandono

In [30]:
churn_percentage_df = (
    df_clean["Churn"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .reset_index()
)

churn_percentage_df.columns = ["Churn", "Porcentaje"]

fig = px.pie(
    churn_percentage_df,
    names="Churn",
    values="Porcentaje",
    title="Porcentaje de clientes que abandonan el servicio"
)

fig.update_traces(
    textinfo="label+percent"
)

fig.show()

### Interpretación

La visualización confirma que la mayoría de los clientes permanecieron en el servicio (73,42%), mientras que el 26,58% corresponde a clientes que abandonaron.

La proporción de clientes que abandonan es suficientemente relevante para justificar el desarrollo de una solución orientada a la identificación temprana de clientes con mayor riesgo de churn.

## 6.3 Churn según tipo de contrato

### Pregunta de negocio

¿Existe una diferencia en la tasa de abandono entre los distintos tipos de contrato?

In [31]:
contract_churn = pd.crosstab(
    df_clean["Contract"],
    df_clean["Churn"],
    normalize="index"
).mul(100).round(2)

contract_churn

Churn,No,Yes
Contract,,
Month-to-month,57.29,42.71
One year,88.72,11.28
Two year,97.15,2.85


In [32]:
contract_churn_rate = (
    df_clean
    .groupby("Contract")["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .round(2)
    .sort_values(ascending=False)
)

contract_churn_rate

,Churn
Contract,
Month-to-month,42.71
One year,11.28
Two year,2.85


In [33]:
contract_churn_plot = (
    df_clean
    .groupby("Contract")["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .reset_index(name="Tasa de abandono (%)")
)

fig = px.bar(
    contract_churn_plot,
    x="Contract",
    y="Tasa de abandono (%)",
    text="Tasa de abandono (%)",
    title="Tasa de abandono según tipo de contrato"
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Tipo de contrato",
    yaxis_title="Tasa de abandono (%)"
)

fig.show()

### Hallazgo

Se observa una diferencia considerable en la tasa de abandono según el tipo de contrato. Los clientes con contrato `Month-to-month` presentan una tasa de abandono de **42,71%**, mientras que los clientes con contrato `One year` presentan un abandono de **11,28%** y aquellos con contrato `Two year` alcanzan solo un **2,85%**.

### Interpretación de negocio

Los clientes con contratos mensuales presentan una tasa de abandono considerablemente mayor que aquellos que poseen contratos de uno o dos años. Este resultado identifica al tipo de contrato como una variable relevante para segmentar el riesgo de abandono.

Desde una perspectiva de negocio, los clientes con modalidad `Month-to-month` podrían representar un segmento prioritario para estrategias de retención.

Sin embargo, este análisis es descriptivo y muestra una asociación entre ambas variables. No permite afirmar que el tipo de contrato sea por sí solo la causa del abandono.

In [34]:
contract_counts = (
    df_clean["Contract"]
    .value_counts()
)

contract_counts

,count
Contract,
Month-to-month,3875
Two year,1685
One year,1472


In [35]:
contract_summary = pd.DataFrame({
    "Clientes": contract_counts,
    "Tasa de abandono (%)": contract_churn_rate
}).sort_values(
    "Tasa de abandono (%)",
    ascending=False
)

contract_summary

,Clientes,Tasa de abandono (%)
Contract,,
Month-to-month,3875,42.71
One year,1472,11.28
Two year,1685,2.85


### Hallazgo: tipo de contrato

La tasa de abandono presenta diferencias importantes según el tipo de contrato.

Los clientes con contrato `Month-to-month` presentan una tasa de abandono de **42,71%**, siendo el segmento con mayor nivel de churn. En contraste, los clientes con contrato `One year` presentan una tasa de **11,28%**, mientras que los clientes con contrato `Two year` presentan una tasa de solo **2,85%**.

Además, el segmento `Month-to-month` corresponde a 3.875 clientes, por lo que no solo presenta la mayor tasa de abandono, sino que también representa una cantidad importante de clientes dentro del conjunto de datos.

### Interpretación de negocio

El tipo de contrato aparece como una variable relevante para segmentar el riesgo de abandono. Los clientes con modalidad mensual constituyen un segmento prioritario para estrategias de retención.

Sin embargo, este resultado corresponde a un análisis descriptivo y no permite afirmar que el tipo de contrato sea la causa directa del abandono.

## 6.4 Churn según antigüedad

### Pregunta de negocio

¿Los clientes con menor antigüedad presentan una mayor tasa de abandono?

In [36]:
tenure_churn_rate = (
    df_clean
    .groupby("tenure")["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .round(2)
)

tenure_churn_rate.head(15)

,Churn
tenure,
1,61.99
2,51.68
3,47.00
4,47.16
5,48.12
6,36.36
7,38.93
8,34.15
9,38.66


In [37]:
fig = px.histogram(
    df_clean,
    x="tenure",
    nbins=30,
    title="Distribución de la antigüedad de los clientes"
)

fig.update_layout(
    xaxis_title="Meses como cliente",
    yaxis_title="Cantidad de clientes",
    bargap=0.1
)

fig.update_traces(marker_line_color='black', marker_line_width=1)

fig.show()

In [38]:
fig = px.box(
    df_clean,
    x="Churn",
    y="tenure",
    points="outliers",
    title="Antigüedad de los clientes según abandono"
)

fig.update_layout(
    xaxis_title="Churn",
    yaxis_title="Antigüedad en meses"
)

fig.show()

### Hallazgo: antigüedad del cliente

La tasa de abandono disminuye a medida que aumenta la antigüedad del cliente.

El grupo de clientes con una antigüedad de `0-12 meses` presenta la mayor tasa de abandono, con **47,68%**. Esta tasa disminuye a **28,71%** para los clientes de `13-24 meses`, a **20,39%** para el grupo de `25-48 meses` y alcanza **9,51%** entre los clientes con `49-72 meses` de antigüedad.

### Interpretación de negocio

Los resultados sugieren que los clientes durante sus primeros meses de relación con la empresa presentan un mayor nivel de riesgo de abandono. Por este motivo, las estrategias de retención podrían concentrarse especialmente en las primeras etapas del ciclo de vida del cliente.

Al igual que en el análisis del tipo de contrato, estos resultados representan asociaciones observadas en los datos y no permiten establecer causalidad por sí solos.

## 6.5 Churn por grupos de antigüedad

In [39]:
bins = [0, 12, 24, 48, 72]

labels = [
    "0-12 meses",
    "13-24 meses",
    "25-48 meses",
    "49-72 meses"
]

df_clean["TenureGroup"] = pd.cut(
    df_clean["tenure"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

df_clean[["tenure", "TenureGroup"]].head()



,tenure,TenureGroup
0,1,0-12 meses
1,34,25-48 meses
2,2,0-12 meses
3,45,25-48 meses
4,2,0-12 meses


In [40]:
tenure_group_churn = (
    df_clean
    .groupby("TenureGroup", observed=False)["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .round(2)
    .sort_values(ascending=False)
)

tenure_group_churn

,Churn
TenureGroup,
0-12 meses,47.68
13-24 meses,28.71
25-48 meses,20.39
49-72 meses,9.51


In [41]:
tenure_churn = (
    df_clean
    .groupby("TenureGroup", observed=False)["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .reset_index(name="Tasa de abandono (%)")
)

tenure_churn

,TenureGroup,Tasa de abandono (%)
0,0-12 meses,47.678161
1,13-24 meses,28.710938
2,25-48 meses,20.388959
3,49-72 meses,9.513176


In [42]:
fig = px.bar(
    tenure_churn,
    x="TenureGroup",
    y="Tasa de abandono (%)",
    text="Tasa de abandono (%)",
    title="Tasa de abandono según antigüedad del cliente"
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Antigüedad",
    yaxis_title="Tasa de abandono (%)"
)

fig.show()

## 6.6 Churn según cargos mensuales

In [43]:
fig = px.histogram(
    df_clean,
    x="MonthlyCharges",
    color="Churn",
    nbins=30,
    barmode="group", # Cambiado de 'overlay' a 'group'
    title="Distribución de cargos mensuales según abandono"
)

fig.update_layout(
    xaxis_title="Cargo mensual",
    yaxis_title="Cantidad de clientes",
    bargap=0.1 # Añadido un espacio entre las barras
)

fig.update_traces(marker_line_color='black', marker_line_width=1) # Añadido borde a las barras

fig.show()

In [44]:
fig = px.box(
    df_clean,
    x="Churn",
    y="MonthlyCharges",
    points="outliers",
    title="Cargos mensuales según abandono"
)

fig.update_layout(
    xaxis_title="Churn",
    yaxis_title="Cargo mensual"
)

fig.show()

In [45]:
monthly_summary = (
    df_clean
    .groupby("Churn")["MonthlyCharges"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

monthly_summary

,count,mean,median,min,max
Churn,,,,,
No,5163,61.31,64.45,18.25,118.75
Yes,1869,74.44,79.65,18.85,118.35


### Interpretación

Se comparan los cargos mensuales entre clientes que permanecieron y clientes que abandonaron. Esta comparación permite evaluar si el nivel de pago mensual presenta diferencias entre ambos grupos.

Los resultados descriptivos permiten identificar posibles diferencias de distribución y detectar si los clientes con mayores cargos mensuales presentan un comportamiento de abandono diferente.

## 6.7 Churn según segmentos de cargos mensuales

In [46]:
df_clean["MonthlyChargesGroup"] = pd.qcut(
    df_clean["MonthlyCharges"],
    q=4,
    duplicates="drop"
)

df_clean[["MonthlyCharges", "MonthlyChargesGroup"]].head()

,MonthlyCharges,MonthlyChargesGroup
0,29.85,"(18.249, 35.588]"
1,56.95,"(35.588, 70.35]"
2,53.85,"(35.588, 70.35]"
3,42.30,"(35.588, 70.35]"
4,70.70,"(70.35, 89.862]"


In [47]:
charges_churn = (
    df_clean
    .groupby("MonthlyChargesGroup", observed=False)["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .reset_index(name="Tasa de abandono (%)")
)

charges_churn

,MonthlyChargesGroup,Tasa de abandono (%)
0,"(18.249, 35.588]",11.319681
1,"(35.588, 70.35]",24.588302
2,"(70.35, 89.862]",37.549858
3,"(89.862, 118.75]",32.878271


In [48]:
charges_churn_str = charges_churn.copy()
charges_churn_str['MonthlyChargesGroup'] = charges_churn_str['MonthlyChargesGroup'].astype(str)

fig = px.bar(
    charges_churn_str,
    x="MonthlyChargesGroup",
    y="Tasa de abandono (%)",
    text="Tasa de abandono (%)",
    title="Tasa de abandono según rango de cargos mensuales"
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.update_layout(
    xaxis_title="Rango de cargos mensuales",
    yaxis_title="Tasa de abandono (%)"
)

fig.show()

### Interpretación

La segmentación de los cargos mensuales permite comparar la tasa de abandono entre grupos de clientes con diferentes niveles de pago mensual.

Este análisis permite identificar si existe un segmento de cargos particularmente asociado con una mayor tasa de abandono y complementa el análisis realizado mediante el histograma y el boxplot.

## 6.8 Churn según servicios contratados

In [49]:
def calcular_tasa_churn(data, columna):
    resultado = (
        data
        .groupby(columna)["Churn"]
        .apply(lambda x: (x == "Yes").mean() * 100)
        .reset_index(name="Tasa de abandono (%)")
        .sort_values("Tasa de abandono (%)", ascending=False)
    )

    return resultado

In [50]:
internet_churn = calcular_tasa_churn(
    df_clean,
    "InternetService"
)

internet_churn

,InternetService,Tasa de abandono (%)
1,Fiber optic,41.892765
0,DSL,18.998344
2,No,7.434211


In [51]:
fig = px.bar(
    internet_churn,
    x="InternetService",
    y="Tasa de abandono (%)",
    text="Tasa de abandono (%)",
    title="Tasa de abandono según servicio de Internet"
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.show()

In [52]:
online_security_churn = calcular_tasa_churn(
    df_clean,
    "OnlineSecurity"
)

fig = px.bar(
    online_security_churn,
    x="OnlineSecurity",
    y="Tasa de abandono (%)",
    text="Tasa de abandono (%)",
    title="Tasa de abandono según servicio de seguridad online"
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.show()

In [53]:
tech_support_churn = calcular_tasa_churn(
    df_clean,
    "TechSupport"
)

tech_support_churn

,TechSupport,Tasa de abandono (%)
0,No,41.647465
2,Yes,15.196078
1,No internet service,7.434211


In [54]:
fig = px.bar(
    tech_support_churn,
    x="TechSupport",
    y="Tasa de abandono (%)",
    text="Tasa de abandono (%)",
    title="Tasa de abandono según soporte técnico"
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.show()

In [55]:
payment_churn = calcular_tasa_churn(
    df_clean,
    "PaymentMethod"
)

payment_churn

,PaymentMethod,Tasa de abandono (%)
2,Electronic check,45.285412
3,Mailed check,19.201995
0,Bank transfer (automatic),16.731518
1,Credit card (automatic),15.253123


In [56]:
fig = px.bar(
    payment_churn,
    x="PaymentMethod",
    y="Tasa de abandono (%)",
    text="Tasa de abandono (%)",
    title="Tasa de abandono según método de pago"
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.show()

In [57]:
paperless_churn = calcular_tasa_churn(
    df_clean,
    "PaperlessBilling"
)

paperless_churn

,PaperlessBilling,Tasa de abandono (%)
1,Yes,33.589251
0,No,16.375698


In [58]:
fig = px.bar(
    paperless_churn,
    x="PaperlessBilling",
    y="Tasa de abandono (%)",
    text="Tasa de abandono (%)",
    title="Tasa de abandono según facturación electrónica"
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside"
)

fig.show()

## Análisis de correlación: ¿Qué variables numéricas se relacionan con el abandono?

Para complementar el análisis exploratorio, se utiliza la correlación de Spearman para identificar qué variables numéricas presentan una mayor asociación con la variable `Churn`.

Se codifica `Churn` como 0 = No y 1 = Sí.

In [59]:

# Copia para no modificar el dataset original
df_corr = df_clean.copy()

# Convertir Churn a variable numérica
df_corr["Churn"] = df_corr["Churn"].map({"No": 0, "Yes": 1})

# Variables numéricas relevantes
variables = ["tenure", "MonthlyCharges", "TotalCharges", "Churn"]

# Correlación de Spearman
spearman_corr = df_corr[variables].corr(method="spearman")

# Mostrar matriz
spearman_corr

,tenure,MonthlyCharges,TotalCharges,Churn
tenure,1.000000,0.275339,0.889177,-0.369659
MonthlyCharges,0.275339,1.000000,0.638032,0.184167
TotalCharges,0.889177,0.638032,1.000000,-0.232190
Churn,-0.369659,0.184167,-0.232190,1.000000


In [63]:
# Mostrar correlaciones con Churn
churn_corr = (
    spearman_corr["Churn"]
    .drop("Churn")
    .sort_values(ascending=False)
)

print("Correlación de las variables numéricas con Churn:")
display(churn_corr)

# Gráfico
fig = go.Figure(
    data=go.Heatmap(
        z=spearman_corr.values,
        x=spearman_corr.columns,
        y=spearman_corr.columns,
        text=spearman_corr.round(2).values,
        texttemplate="%{text}",
        colorscale="RdBu",
        zmin=-1,
        zmax=1
    )
)

fig.update_layout(
    title="Matriz de correlación de Spearman",
    xaxis_title="Variables",
    yaxis_title="Variables"
)

fig.show()

Correlación de las variables numéricas con Churn:


,Churn
MonthlyCharges,0.184167
TotalCharges,-0.232190
tenure,-0.369659


### Hallazgo

La correlación de Spearman permite identificar las variables numéricas que presentan mayor asociación con el abandono. Los resultados muestran que **MonthlyCharges** y **tenure** presentan relaciones relevantes con `Churn`.

Sin embargo, la correlación **no implica causalidad**. Estos resultados permiten identificar patrones que pueden ser útiles posteriormente para construir un modelo predictivo de abandono.


## Análisis de asociación entre variables categóricas

### Pregunta
¿Existe una asociación entre las variables categóricas y el abandono de clientes (`Churn`)?

Para analizarlo utilizaremos **V de Cramér**, una medida de asociación entre variables categóricas que toma valores entre 0 y 1.

In [64]:
from scipy.stats import chi2_contingency

# Variables categóricas, excluyendo customerID
categorical_columns = df_clean.select_dtypes(include="object").columns.tolist()
categorical_columns.remove("customerID")

# Función para calcular V de Cramér
def cramers_v(x, y):
    tabla = pd.crosstab(x, y)
    chi2 = chi2_contingency(tabla)[0]
    n = tabla.sum().sum()
    r, k = tabla.shape

    phi2 = chi2 / n
    phi2_corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    r_corr = r - ((r - 1) ** 2) / (n - 1)
    k_corr = k - ((k - 1) ** 2) / (n - 1)

    return np.sqrt(
        phi2_corr / min(k_corr - 1, r_corr - 1)
    )

# Crear matriz
cramers_matrix = pd.DataFrame(
    index=categorical_columns,
    columns=categorical_columns,
    dtype=float
)

for col1 in categorical_columns:
    for col2 in categorical_columns:
        cramers_matrix.loc[col1, col2] = cramers_v(
            df_clean[col1],
            df_clean[col2]
        )

# Mostrar matriz
cramers_matrix.round(2)

,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Churn
gender,1.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Partner,0.0,1.00,0.45,0.01,0.14,0.00,0.15,0.15,0.17,0.13,0.14,0.13,0.29,0.01,0.16,0.15
Dependents,0.0,0.45,1.00,0.00,0.02,0.18,0.19,0.16,0.16,0.18,0.14,0.14,0.24,0.11,0.15,0.16
PhoneService,0.0,0.01,0.00,1.00,1.00,0.45,0.17,0.17,0.17,0.18,0.18,0.18,0.00,0.01,0.00,0.00
MultipleLines,0.0,0.14,0.02,1.00,1.00,0.40,0.23,0.24,0.25,0.23,0.26,0.26,0.08,0.17,0.17,0.04
InternetService,0.0,0.00,0.18,0.45,0.40,1.00,0.72,0.71,0.71,0.72,0.72,0.72,0.21,0.38,0.31,0.32
OnlineSecurity,0.0,0.15,0.19,0.17,0.23,0.72,1.00,0.72,0.72,0.73,0.71,0.71,0.30,0.34,0.30,0.35
OnlineBackup,0.0,0.15,0.16,0.17,0.24,0.71,0.72,1.00,0.72,0.72,0.71,0.71,0.26,0.32,0.28,0.29
DeviceProtection,0.0,0.17,0.16,0.17,0.25,0.71,0.72,0.72,1.00,0.73,0.73,0.74,0.30,0.32,0.29,0.28
TechSupport,0.0,0.13,0.18,0.18,0.23,0.72,0.73,0.72,0.73,1.00,0.72,0.72,0.33,0.33,0.31,0.34


In [65]:
fig = go.Figure(
    data=go.Heatmap(
        z=cramers_matrix.values,
        x=cramers_matrix.columns,
        y=cramers_matrix.columns,
        text=cramers_matrix.round(2).values,
        texttemplate="%{text}",
        colorscale="RdBu",
        zmin=0,
        zmax=1,
        colorbar=dict(title="V de Cramér")
    )
)

fig.update_layout(
    title="Matriz de asociación de variables categóricas - V de Cramér",
    xaxis_title="Variables categóricas",
    yaxis_title="Variables categóricas",
    width=900,
    height=800
)

fig.show()

In [66]:
churn_cramers = (
    cramers_matrix["Churn"]
    .drop("Churn")
    .sort_values(ascending=False)
)

print("Asociación de las variables categóricas con Churn:")
display(churn_cramers.round(2))

Asociación de las variables categóricas con Churn:


,Churn
Contract,0.41
OnlineSecurity,0.35
TechSupport,0.34
InternetService,0.32
PaymentMethod,0.30
OnlineBackup,0.29
DeviceProtection,0.28
StreamingMovies,0.23
StreamingTV,0.23
PaperlessBilling,0.19


### Hallazgo

La matriz de V de Cramér permite identificar asociaciones entre las variables categóricas. Al analizar específicamente su relación con `Churn`, se observan diferencias en la intensidad de asociación entre las distintas variables.

Las variables con mayor asociación pueden considerarse relevantes para continuar con el proceso de modelamiento. Sin embargo, una asociación no implica causalidad.

# 7. Evaluación de sesgos, ética y privacidad

## 7.1 Posibles sesgos

El conjunto de datos contiene variables demográficas como `gender`, `SeniorCitizen`, `Partner` y `Dependents`. Estas variables podrían generar diferencias en las predicciones entre distintos grupos.

Por este motivo, durante el desarrollo del modelo se deberá evaluar el desempeño por segmentos y evitar utilizar las variables demográficas de manera que generen decisiones injustificadamente discriminatorias.

Además, las diferencias observadas en el EDA representan asociaciones estadísticas y no deben interpretarse como características causales de determinados grupos.

## 7.2 Privacidad

El dataset contiene un identificador de cliente (`customerID`). Esta variable permite identificar registros individuales, pero no aporta información predictiva útil para determinar el abandono.

Por razones de privacidad y para evitar que el modelo aprenda identificadores sin significado, `customerID` no deberá utilizarse como variable predictora durante el modelamiento.

## 7.3 Uso responsable del modelo

El modelo debe utilizarse como una herramienta de apoyo para identificar clientes con mayor riesgo de abandono y orientar estrategias de retención.

Una predicción de alto riesgo no debe utilizarse automáticamente para tomar decisiones negativas sobre un cliente. Las decisiones comerciales deben considerar información adicional y mecanismos de revisión.

# 8. Conclusiones del análisis exploratorio

El análisis exploratorio permitió identificar diferencias relevantes en la tasa de abandono según distintas características de los clientes.

La tasa general de abandono obtenida fue de **26,58%**, estableciendo una línea base para el problema.

Los clientes con contrato `Month-to-month` presentaron una tasa de abandono de **42,71%**, considerablemente superior a los contratos de uno y dos años.

También se observó una relación descriptiva entre la antigüedad y el abandono: los clientes con `0-12 meses` presentaron una tasa de abandono de **47,68%**, mientras que el grupo de `49-72 meses` presentó un **9,51%**.

Estos resultados permiten identificar segmentos que deberían recibir especial atención en las estrategias de retención. Sin embargo, las asociaciones observadas no implican causalidad y deberán ser evaluadas posteriormente mediante un modelo de Machine Learning.

Los hallazgos obtenidos servirán como base para la siguiente etapa del proyecto: la preparación de las variables para el modelamiento predictivo.

# 9. Metodología CRISP-DM

El proyecto sigue la metodología CRISP-DM:

1. **Comprensión del negocio:** identificar el problema de abandono de clientes y establecer objetivos y KPIs.
2. **Comprensión de los datos:** revisar estructura, tipos de datos, calidad y distribución de las variables.
3. **Preparación de los datos:** limpiar valores inconsistentes y preparar el dataset para análisis posterior.
4. **Modelamiento:** corresponde a una etapa posterior del proyecto y no forma parte del alcance de esta evaluación.
5. **Evaluación:** será realizada en una etapa posterior utilizando métricas apropiadas.
6. **Despliegue:** corresponde a una etapa futura, una vez validada la solución.

## Fuente de datos y herramientas

### Fuente de datos

Se utiliza el dataset **Telco Customer Churn**, disponible en Kaggle. El conjunto contiene información demográfica, servicios contratados, información de cuenta, antigüedad, cargos y la variable `Churn`.

### Herramientas utilizadas

- **Python:** manipulación, limpieza y análisis de los datos.
- **Pandas:** procesamiento del dataset.
- **NumPy:** operaciones numéricas.
- **Plotly:** generación de visualizaciones interactivas.
- **Google Colab:** desarrollo y ejecución del notebook.
- **Kaggle:** fuente del conjunto de datos.
- **Git/GitHub:** organización y colaboración del proyecto.